In [1]:
import pickle
import numpy as np
import tensorflow as tf
import keras
from cnn_base import build_t1, build_t1_quantizade
from utils import load_pickle_data
import os


TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
DATA_DIR = "./dataset/sat-6-full.mat"
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

2026-10-31 15:38:21.564159: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-31 15:38:21.565300: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-31 15:38:21.585151: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-10-31 15:38:21.585834: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-31 15:38:21.924317: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [2]:
X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32", copy=False)
X_train_all /= 256.0
X_test = X_test.astype("float32", copy=False)
X_test /= 256.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)
input_shape, train_size

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


((32, 32, 4), 243000)

# Train T1 base model

In [3]:
# Finn base model not quantized
model = build_t1()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1,),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks,batch_size=BATCH_SIZE)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1', exist_ok=True)
base_path = './models/t1/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 3s 5ms/step - loss: 0.5387 - accuracy: 0.7866 - val_loss: 0.2843 - val_accuracy: 0.8906
Epoch 2/30
475/475 [==============================] - 2s 4ms/step - loss: 0.2439 - accuracy: 0.9103 - val_loss: 0.2032 - val_accuracy: 0.9284
Epoch 3/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1799 - accuracy: 0.9333 - val_loss: 0.1586 - val_accuracy: 0.9413
Epoch 4/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1491 - accuracy: 0.9433 - val_loss: 0.1399 - val_accuracy: 0.9454
Epoch 5/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1352 - accuracy: 0.9474 - val_loss: 0.1291 - val_accuracy: 0.9485
Epoch 6/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1253 - accuracy: 0.9509 - val_loss: 0.1203 - val_accuracy: 0.9523
Epoch 7/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1180 - accuracy: 0.9532 - val_loss: 0.1145 - val_accuracy: 0.9539
Epoch 

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [4]:
# Finn base model not quantized with max pooling and batch normalization
model = build_t1(max_pool=True, batch_norm=True)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks, batch_size=BATCH_SIZE)


os.makedirs('models', exist_ok=True)
os.makedirs('models/t1_maxpool_bn', exist_ok=True)
base_path = './models/t1_maxpool_bn/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 3s 6ms/step - loss: 0.5043 - accuracy: 0.8423 - val_loss: 0.3190 - val_accuracy: 0.9354
Epoch 2/30
475/475 [==============================] - 2s 4ms/step - loss: 0.1469 - accuracy: 0.9535 - val_loss: 0.1234 - val_accuracy: 0.9604
Epoch 3/30
475/475 [==============================] - 2s 5ms/step - loss: 0.1079 - accuracy: 0.9637 - val_loss: 0.0948 - val_accuracy: 0.9684
Epoch 4/30
475/475 [==============================] - 2s 5ms/step - loss: 0.0890 - accuracy: 0.9699 - val_loss: 0.0845 - val_accuracy: 0.9725
Epoch 5/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0769 - accuracy: 0.9740 - val_loss: 0.0762 - val_accuracy: 0.9747
Epoch 6/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0685 - accuracy: 0.9766 - val_loss: 0.0674 - val_accuracy: 0.9777
Epoch 7/30
475/475 [==============================] - 2s 4ms/step - loss: 0.0630 - accuracy: 0.9785 - val_loss: 0.0710 - val_accuracy: 0.9739
Epoch 

# Finn Model

In [5]:
bits = [8, 4]

for bit in bits:
    # entrada x/256 -> ap_ufixed<8,0> (uint8). Para int8 com sinal: signed_input=True e X = (x-128)/128
    model = build_t1_quantizade(
        input_shape=input_shape,
        n_classes=n_classes,
        bits=bit,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
        #keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks, batch_size=BATCH_SIZE)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 9s 18ms/step - loss: 0.4126 - accuracy: 0.8646 - val_loss: 0.1617 - val_accuracy: 0.9457
Epoch 2/30
475/475 [==============================] - 8s 17ms/step - loss: 0.1374 - accuracy: 0.9544 - val_loss: 0.1166 - val_accuracy: 0.9619
Epoch 3/30
475/475 [==============================] - 8s 17ms/step - loss: 0.1091 - accuracy: 0.9632 - val_loss: 0.1015 - val_accuracy: 0.9673
Epoch 4/30
475/475 [==============================] - 8s 17ms/step - loss: 0.0966 - accuracy: 0.9675 - val_loss: 0.0916 - val_accuracy: 0.9698
Epoch 5/30
475/475 [==============================] - 8s 17ms/step - loss: 0.0892 - accuracy: 0.9692 - val_loss: 0.0832 - val_accuracy: 0.9721
Epoch 6/30
475/475 [==============================] - 8s 17ms/step - loss: 0.0829 - accuracy: 0.9710 - val_loss: 0.0790 - val_accuracy: 0.9720
Epoch 7/30
475/475 [==============================] - 8s 17ms/step - loss: 0.0783 - accuracy: 0.9727 - val_loss: 0.0750 - val_accuracy: 0.9738

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer HeNormal is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


Epoch 1/30
475/475 [==============================] - 9s 18ms/step - loss: 0.4546 - accuracy: 0.8259 - val_loss: 0.2557 - val_accuracy: 0.9014
Epoch 2/30
475/475 [==============================] - 8s 17ms/step - loss: 0.2467 - accuracy: 0.9054 - val_loss: 0.2110 - val_accuracy: 0.9251
Epoch 3/30
475/475 [==============================] - 8s 17ms/step - loss: 0.2144 - accuracy: 0.9213 - val_loss: 0.2015 - val_accuracy: 0.9209
Epoch 4/30
475/475 [==============================] - 8s 17ms/step - loss: 0.1769 - accuracy: 0.9382 - val_loss: 0.1505 - val_accuracy: 0.9488
Epoch 5/30
475/475 [==============================] - 8s 18ms/step - loss: 0.1500 - accuracy: 0.9480 - val_loss: 0.1553 - val_accuracy: 0.9415
Epoch 6/30
475/475 [==============================] - 8s 17ms/step - loss: 0.1445 - accuracy: 0.9472 - val_loss: 0.1247 - val_accuracy: 0.9561
Epoch 7/30
475/475 [==============================] - 8s 17ms/step - loss: 0.1297 - accuracy: 0.9530 - val_loss: 0.1182 - val_accuracy: 0.9572

# Finn Model with pooling

In [6]:
bits = [8, 4]

for bit in bits:
    model = build_t1_quantizade(
        input_shape=input_shape,
        n_classes=n_classes,
        bits=bit,
        max_pool=True,
        batch_norm=True,
    )

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
        metrics=[
            tf.keras.metrics.CategoricalAccuracy(name="accuracy")
        ],
    )
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
       # keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ]

    model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks, batch_size=BATCH_SIZE)


    os.makedirs('models', exist_ok=True)
    model_name = f't1_{bit}_maxpool_bn'
    os.makedirs(f'models/{model_name}', exist_ok=True)
    base_path = f'./models/{model_name}/'
    model.save(base_path + '/model.keras')
    model.save(base_path + '/model.h5')
    model.save_weights(base_path + '/weights.h5')

    with open(base_path + '/model.json', 'w') as f:
        f.write(model.to_json())

Epoch 1/30
475/475 [==============================] - 12s 24ms/step - loss: 0.5789 - accuracy: 0.8224 - val_loss: 0.3244 - val_accuracy: 0.8990
Epoch 2/30
475/475 [==============================] - 11s 24ms/step - loss: 0.2222 - accuracy: 0.9427 - val_loss: 0.2158 - val_accuracy: 0.9278
Epoch 3/30
475/475 [==============================] - 11s 24ms/step - loss: 0.1472 - accuracy: 0.9593 - val_loss: 0.1300 - val_accuracy: 0.9631
Epoch 4/30
475/475 [==============================] - 11s 24ms/step - loss: 0.1144 - accuracy: 0.9674 - val_loss: 0.1086 - val_accuracy: 0.9687
Epoch 5/30
475/475 [==============================] - 11s 24ms/step - loss: 0.0966 - accuracy: 0.9715 - val_loss: 0.0889 - val_accuracy: 0.9740
Epoch 6/30
475/475 [==============================] - 11s 24ms/step - loss: 0.0847 - accuracy: 0.9740 - val_loss: 0.0796 - val_accuracy: 0.9751
Epoch 7/30
475/475 [==============================] - 11s 24ms/step - loss: 0.0777 - accuracy: 0.9753 - val_loss: 0.0713 - val_accuracy: